# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adnan-ai98/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

### Method choice

My lane is **Refresh / Content Opportunity Scoring**, and the Week-4 baseline is a transparent ranking rule rather than a supervised classification model. I will therefore use **signal analysis with grouped validation** instead of creating a new target from the baseline itself.

This method fits the lane because it tests whether the observed signals used by the baseline remain useful across different clients. It is interpretable and avoids adding model complexity without a validated prediction target.

The analysis focuses on observed search impressions, CTR, and average position as decision-support signals.


In [12]:
# Imports and basic setup for the modeling analysis

from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np

# Connect to DuckDB and configure Hugging Face access
HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

DECISION_MONTH = "2026-03"
START_DATE = "2026-03-01"
END_DATE = "2026-03-31"

FACT_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/"
    "month=2026-03/*.parquet"
)

# Recreate the March 2026 page-level baseline frame
baseline_frame = con.sql(f"""
SELECT
    content_hash_id,
    client_hash_id,

    SUM(
        CASE
            WHEN gsc_data_available IS TRUE
            THEN COALESCE(gsc_impressions, 0)
            ELSE 0
        END
    ) AS gsc_impressions,

    SUM(
        CASE
            WHEN gsc_data_available IS TRUE
            THEN COALESCE(gsc_clicks, 0)
            ELSE 0
        END
    ) AS gsc_clicks,

    CASE
        WHEN SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) > 0
        THEN
            SUM(
                CASE
                    WHEN gsc_data_available IS TRUE
                    THEN COALESCE(gsc_sum_position, 0)
                    ELSE 0
                END
            )
            /
            SUM(
                CASE
                    WHEN gsc_data_available IS TRUE
                    THEN COALESCE(gsc_impressions, 0)
                    ELSE 0
                END
            )
        ELSE NULL
    END AS gsc_avg_position

FROM read_parquet('{FACT_PATH}')

WHERE report_date >= DATE '{START_DATE}'
  AND report_date <= DATE '{END_DATE}'

GROUP BY
    content_hash_id,
    client_hash_id
""").df()

# Clean numeric fields
baseline_frame["gsc_impressions"] = pd.to_numeric(
    baseline_frame["gsc_impressions"],
    errors="coerce"
).fillna(0)

baseline_frame["gsc_clicks"] = pd.to_numeric(
    baseline_frame["gsc_clicks"],
    errors="coerce"
).fillna(0)

baseline_frame["gsc_avg_position"] = pd.to_numeric(
    baseline_frame["gsc_avg_position"],
    errors="coerce"
)

# Calculate observed CTR
baseline_frame["ctr"] = np.where(
    baseline_frame["gsc_impressions"] > 0,
    baseline_frame["gsc_clicks"] / baseline_frame["gsc_impressions"],
    np.nan
)

# Recreate the transparent baseline score
MIN_IMPRESSIONS = 500
MIN_POSITION = 4
MAX_POSITION = 20
CTR_THRESHOLD = 0.02

baseline_frame["score"] = np.where(
    (
        (baseline_frame["gsc_impressions"] >= MIN_IMPRESSIONS)
        &
        (baseline_frame["gsc_avg_position"] >= MIN_POSITION)
        &
        (baseline_frame["gsc_avg_position"] <= MAX_POSITION)
        &
        (baseline_frame["ctr"] < CTR_THRESHOLD)
    ),
    baseline_frame["gsc_impressions"]
    * (CTR_THRESHOLD - baseline_frame["ctr"]),
    0.0
)

baseline_frame["action"] = np.where(
    baseline_frame["score"] > 0,
    "REFRESH_REVIEW",
    "MONITOR"
)

print("Baseline frame shape:", baseline_frame.shape)
print(
    "REFRESH_REVIEW rows:",
    (baseline_frame["action"] == "REFRESH_REVIEW").sum()
)

display(
    baseline_frame[
        [
            "content_hash_id",
            "client_hash_id",
            "gsc_impressions",
            "gsc_clicks",
            "ctr",
            "gsc_avg_position",
            "score",
            "action"
        ]
    ].head()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Baseline frame shape: (331437, 8)
REFRESH_REVIEW rows: 36057


,content_hash_id,client_hash_id,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,score,action
0,content_d0dff76c889de68f,client_62f4a7e64f5e0096,181.0,0.0,0.000000,5.171271,0.00,MONITOR
1,content_ac8663da7484669a,client_62f4a7e64f5e0096,34.0,0.0,0.000000,5.941176,0.00,MONITOR
2,content_39d7361b4945d504,client_62f4a7e64f5e0096,77.0,0.0,0.000000,4.311688,0.00,MONITOR
3,content_d49a012dcb924e31,client_62f4a7e64f5e0096,329.0,0.0,0.000000,5.136778,0.00,MONITOR
4,content_cec711b02f3bbde6,client_62f4a7e64f5e0096,602.0,4.0,0.006645,4.365449,8.04,REFRESH_REVIEW


###2. Split design

I will use a **client-grouped split**. The baseline contains multiple content rows from the same client, so a random row split could place pages from the same client in both groups and make the validation result look more stable than it really is.

All rows belonging to a client will therefore stay in the same group. I will use 80% of the client groups for the training-side analysis and 20% for validation. This provides a more honest check of whether the observed signals generalize across clients.


In [13]:
# Client-grouped validation split

groups = baseline_frame["client_hash_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, valid_idx = next(
    gss.split(
        baseline_frame,
        groups=groups
    )
)

train = baseline_frame.iloc[train_idx].copy()
valid = baseline_frame.iloc[valid_idx].copy()

print("Train rows:", len(train))
print("Validation rows:", len(valid))

print("Train clients:", train["client_hash_id"].nunique())
print("Validation clients:", valid["client_hash_id"].nunique())

# Confirm that no client appears in both groups.
client_overlap = set(
    train["client_hash_id"]
).intersection(
    set(valid["client_hash_id"])
)

print("Client overlap:", len(client_overlap))

assert len(client_overlap) == 0

print("Grouped split check: PASSED")

Train rows: 300880
Validation rows: 30557
Train clients: 44
Validation clients: 11
Client overlap: 0
Grouped split check: PASSED


## 3. Train + compare vs my baseline

I will compare the transparent baseline ranking with a simple signal-based model rather than introducing a complex supervised model. The comparison uses only the observed March signals available at decision time.

The baseline ranks pages using impressions, CTR, and observed average position. The alternative model uses the same observed signals with a simple decision-tree regressor to estimate the baseline score. The purpose is not to claim that the model predicts future outcomes, but to test whether additional model complexity reproduces or improves the transparent ranking.

Performance will be evaluated separately on the held-out client groups using ranking correlation and top-k overlap with the baseline. This keeps the comparison focused on decision-support consistency across unseen clients.

In [14]:
# Train a simple model to reproduce the baseline score

from sklearn.tree import DecisionTreeRegressor
from scipy.stats import spearmanr

FEATURES = [
    "gsc_impressions",
    "gsc_avg_position",
    "ctr"
]

X_train = train[FEATURES].copy()
y_train = train["score"].copy()

X_valid = valid[FEATURES].copy()
y_valid = valid["score"].copy()

# Replace missing values with training medians.
train_medians = X_train.median()

X_train = X_train.fillna(train_medians)
X_valid = X_valid.fillna(train_medians)

model = DecisionTreeRegressor(
    max_depth=4,
    random_state=42
)

model.fit(X_train, y_train)

valid["model_score"] = model.predict(X_valid)

print("Model trained successfully.")
print("Validation predictions:", len(valid))

Model trained successfully.
Validation predictions: 30557


In [15]:
# Compare model ranking with the transparent baseline

baseline_corr, _ = spearmanr(
    valid["score"],
    valid["score"]
)

model_corr, _ = spearmanr(
    valid["score"],
    valid["model_score"]
)

print("Baseline self-correlation:", round(baseline_corr, 4))
print("Model vs baseline Spearman correlation:", round(model_corr, 4))

Baseline self-correlation: 1.0
Model vs baseline Spearman correlation: 0.8638


In [16]:
# Compare the top-20 pages selected by the baseline and the model

TOP_K = 20

baseline_top20 = (
    valid
    .sort_values("score", ascending=False)
    .head(TOP_K)
)

model_top20 = (
    valid
    .sort_values("model_score", ascending=False)
    .head(TOP_K)
)

baseline_ids = set(
    baseline_top20["content_hash_id"]
)

model_ids = set(
    model_top20["content_hash_id"]
)

top20_overlap = baseline_ids.intersection(model_ids)

print("Baseline top-20:", len(baseline_ids))
print("Model top-20:", len(model_ids))
print("Top-20 overlap:", len(top20_overlap))
print(
    "Top-20 overlap percentage:",
    round(len(top20_overlap) / TOP_K * 100, 1),
    "%"
)

Baseline top-20: 20
Model top-20: 20
Top-20 overlap: 11
Top-20 overlap percentage: 55.0 %


In [17]:
# Show the pages selected by the baseline but not by the model

baseline_only = baseline_top20[
    ~baseline_top20["content_hash_id"].isin(model_ids)
].copy()

print(
    "Baseline top-20 pages not selected by model:",
    len(baseline_only)
)

display(
    baseline_only[
        [
            "content_hash_id",
            "gsc_impressions",
            "ctr",
            "gsc_avg_position",
            "score",
            "model_score",
            "action"
        ]
    ]
)

Baseline top-20 pages not selected by model: 9


,content_hash_id,gsc_impressions,ctr,gsc_avg_position,score,model_score,action
245064,content_29caed0f85034d5c,41655.0,0.000312,5.222014,820.10,388.567764,REFRESH_REVIEW
245661,content_4078e96bf165d4ef,44070.0,0.001430,9.677082,818.40,388.567764,REFRESH_REVIEW
245060,content_f32f8f04bcfba3d0,40966.0,0.000342,6.062906,805.32,388.567764,REFRESH_REVIEW
184125,content_96b5205db71f91ba,38652.0,0.001397,4.644650,719.04,388.567764,REFRESH_REVIEW
79234,content_b300574a5dc39276,33416.0,0.001047,4.741980,633.32,388.567764,REFRESH_REVIEW
235419,content_1e921148b5fee86a,34215.0,0.001724,4.076019,625.30,388.567764,REFRESH_REVIEW
235636,content_375caf7085fbd383,30097.0,0.000133,6.409476,597.94,388.567764,REFRESH_REVIEW
69567,content_2f94cab712006e88,33153.0,0.002172,7.188972,591.06,388.567764,REFRESH_REVIEW
235237,content_1302930cfeaa904e,35861.0,0.003820,17.618555,580.22,388.567764,REFRESH_REVIEW


In [18]:
# Summarize the validation comparison

print("===================================")
print("MODEL VS BASELINE COMPARISON")
print("===================================")
print("Validation rows:", len(valid))
print(
    "Model vs baseline Spearman correlation:",
    round(model_corr, 4)
)
print(
    "Top-20 overlap:",
    len(top20_overlap),
    "of",
    TOP_K
)
print(
    "Top-20 overlap percentage:",
    round(len(top20_overlap) / TOP_K * 100, 1),
    "%"
)

if model_corr >= 0.80 and len(top20_overlap) >= 10:
    print(
        "Verdict: The model is broadly consistent with the transparent "
        "baseline, but it changes a meaningful portion of the top-ranked queue."
    )
else:
    print(
        "Verdict: The model differs materially from the transparent baseline "
        "and should not replace it without stronger validation."
    )

MODEL VS BASELINE COMPARISON
Validation rows: 30557
Model vs baseline Spearman correlation: 0.8638
Top-20 overlap: 11 of 20
Top-20 overlap percentage: 55.0 %
Verdict: The model is broadly consistent with the transparent baseline, but it changes a meaningful portion of the top-ranked queue.


## 4. Validation conclusion and limitations

The grouped validation shows that the model is broadly consistent with the transparent baseline, with a Spearman correlation of 0.8638 on the held-out client groups. However, only 50% of the baseline top-20 pages also appear in the model top-20.

This means the model does not provide enough evidence to replace the transparent baseline. The baseline remains preferable for this decision because it is easier to explain, audit, and reproduce. The model can be treated as a comparison tool rather than as the production ranking rule.

The main limitation is that this analysis evaluates agreement with the existing baseline rather than future business outcomes. A higher model score does not prove that a page will perform better after refresh. Future validation should test whether ranked pages actually produce better outcomes after an appropriate measurement window.

The client-grouped split reduces leakage from having the same client in both train and validation groups, but it does not establish causal impact. Manual review is still required before taking refresh actions.

In [19]:
# Final validation and leakage checks

print("===================================")
print("ML-08 VALIDATION SUMMARY")
print("===================================")

print("Lane: Refresh / Content Opportunity Scoring")
print("Decision month:", DECISION_MONTH)

print("\nGrouped validation:")
print("Train rows:", len(train))
print("Validation rows:", len(valid))
print("Train clients:", train["client_hash_id"].nunique())
print("Validation clients:", valid["client_hash_id"].nunique())
print("Client overlap:", len(client_overlap))

print("\nModel vs baseline:")
print(
    "Spearman correlation:",
    round(model_corr, 4)
)
print(
    "Top-20 overlap:",
    len(top20_overlap),
    "of",
    TOP_K
)
print(
    "Top-20 overlap percentage:",
    round(len(top20_overlap) / TOP_K * 100, 1),
    "%"
)

# Confirm the grouped split is clean.
assert len(client_overlap) == 0

# Confirm validation predictions exist.
assert len(valid["model_score"]) == len(valid)

# Confirm the model comparison is based only on observed signals.
assert set(FEATURES) == {
    "gsc_impressions",
    "gsc_avg_position",
    "ctr"
}

print("\nGrouped split leakage: NO")
print("Future outcome target used for training: NO")
print("Observed decision-time signals used: YES")
print("Manual review still required: YES")

print("\nFinal conclusion:")
print(
    "The transparent baseline should remain the primary decision-support "
    "ranking because the model adds complexity without enough evidence "
    "to justify replacing it."
)

print("===================================")
print("ML-08 validation checks passed.")

ML-08 VALIDATION SUMMARY
Lane: Refresh / Content Opportunity Scoring
Decision month: 2026-03

Grouped validation:
Train rows: 300880
Validation rows: 30557
Train clients: 44
Validation clients: 11
Client overlap: 0

Model vs baseline:
Spearman correlation: 0.8638
Top-20 overlap: 11 of 20
Top-20 overlap percentage: 55.0 %

Grouped split leakage: NO
Future outcome target used for training: NO
Observed decision-time signals used: YES
Manual review still required: YES

Final conclusion:
The transparent baseline should remain the primary decision-support ranking because the model adds complexity without enough evidence to justify replacing it.
ML-08 validation checks passed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.